# Sistema de traducción automática con algoritmos bioinspirados

## DATASET LATIN - INGLES

In [94]:
import re
import unicodedata
import numpy as np
import pandas as pd
import sentencepiece as spm
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
from sklearn.model_selection import train_test_split

dataset_latin = load_dataset("grosenthal/latin_english_translation")

df = pd.DataFrame(dataset_latin["train"])
#train_df.to_csv("latin_english_train.csv", index=False)



In [95]:
df.head()

,id,la,en,file
0,81997,post haec ambulabat Iesus in Galilaeam non enim volebat in Iudaeam ambulare quia quaerebant eum Iudaei interficere,"After these things, Jesus walked in Galilee: for he would not walk in Judea, because the Jews sought to kill him.",final_alignments\Vulgate_Bible.json
1,65040,"Despectus tibi sum nec, qui sim, quaeris, Alexi, quam dives pecoris, nivei quam lactis abundans: mille meae Siculis errant in montibus agnae; lac mihi non aestate novum, non frigore defit.","You scorn me, Alexis, and ask not what I am'how rich in cattle, how wealthy in snow-white milk! A thousand lambs of mine roam over the Sicilian hills; new milk fails me not, summer or winter.",final_alignments\Virgil_Eclogues.json
2,15899,"frumenta non solum tanta multitudine iumentorum atque hominum consumebantur, sed etiam anni tempore atque imbribus procubuerant ut, si qui etiam in praesentia se occultassent, tamen his deducto exercitu rerum omnium inopia pereundum videretur.","the corn not only was being consumed by so great numbers of cattle and men, but also had fallen to the earth, owing to the time of the year and the storms; so that if any had concealed themselves for the present, still, it appeared likely that they must perish through want of all things, when the army should be drawn off.",final_alignments\Caesar_DBG_Book6.json
3,38602,Talibus hic Poenus; sed contra Oenotria pubes non ullas voces ducis aut praecepta requirit.,"Thus spoke Hannibal; but the men of Rome, on their side, needed no speech or appeal from any leader.",final_alignments\Silius_Italicus_Punica_Book12.json
4,29514,"non tulit ulterius capti blasphemia monstri Virtutum regina Fides, sed verba loquentis inpedit et vocis claudit spiramina pilo, pollutam rigida transfigens cuspide linguam.","No further did Faith, the Virtues queen, bear with the outrageous prisoners blasphemies, but stopped her speech and blocked the passage of her voice with a javelin, driving its hard point through the foul tongue.",final_alignments\Prudentius_Psychomachia.json


In [96]:
df.shape

(99343, 4)

## DATASET GRIEGO - INGLES

In [97]:

dataset_griego= load_dataset("alansato/greek-english-dataset")

df_g = pd.DataFrame(dataset_griego)


In [98]:
df_g.head()

pd.set_option('display.max_columns', None)     # Mostrar todas las columnas
pd.set_option('display.max_colwidth', None)    # Mostrar texto completo en cada celda
pd.set_option('display.width', None)           # Ajustar ancho automáticamente

df_g.head()

df_nuevo = pd.json_normalize(df_g['train'])

df_nuevo.reset_index(inplace=True)
df_nuevo.rename(columns={'index': 'id'}, inplace=True)

df_nuevo.head()

C:\Users\jeffr\AppData\Local\Temp\ipykernel_19512\783770887.py:11: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_nuevo.reset_index(inplace=True)
C:\Users\jeffr\AppData\Local\Temp\ipykernel_19512\783770887.py:12: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_nuevo.rename(columns={'index': 'id'}, inplace=True)


,id,English,Greek
0,0,Resumption of the session,Επαvάληψη της συvσδoυ
1,1,"I declare resumed the session of the European Parliament adjourned on Friday 17 December 1999, and I would like once again to wish you a happy new year in the hope that you enjoyed a pleasant festive period.","Κηρύσσω την επανάληψη της συνόδου του Ευρωπαϊκού Κοινοβουλίου η οποία είχε διακοπεί την Παρασκευή 17 Δεκεμβρίου και σας απευθύνω ξανά τις θερμές ευχές μου, ελπίζοντας να περάσατε καλά στις διακοπές."
2,2,"Although, as you will have seen, the dreaded 'millennium bug' failed to materialise, still the people in a number of countries suffered a series of natural disasters that truly were dreadful.","Όπως μπορέσατε να διαπιστώσετε, ο περίφημος ""ιός του έτους 2000"" δεν εμφανίσθηκε. Αντιθέτως, οι πολίτες ορισμένων χωρών μας υπήρξαν θύματα φυσικών καταστροφών, οι οποίες ήταν όντως φοβερές."
3,3,"You have requested a debate on this subject in the course of the next few days, during this part-session.","Επιθυμείτε μία συζήτηση επί του θέματος τις επόμενες ημέρες, κατά τη διάρκεια της τρέχουσας περιόδου συνόδου."
4,4,"In the meantime, I should like to observe a minute' s silence, as a number of Members have requested, on behalf of all the victims concerned, particularly those of the terrible storms, in the various countries of the European Union.","Επί του παρόντος θα ήθελα, όπως μου ζήτησαν ορισμένοι συνάδελφοι, να τηρήσουμε ενός λεπτού σιγή για όλα τα θύματα, κυρίως για τα θύματα των καταιγίδων, στις διάφορες πληγείσες χώρες της Ευρωπαϊκής Ένωσης."


### Pipeline de Normalización y División


In [99]:
def normalizar_texto(texto):
    texto = unicodedata.normalize("NFC", str(texto))
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto.lower()

In [ ]:


# 1. Normalizar Latin-English
df_latin_norm = df[['la', 'en']].copy()
df_latin_norm.rename(columns={'la': 'objetivo', 'en': 'fuente'}, inplace=True)
df_latin_norm['fuente'] = df_latin_norm['fuente'].map(normalizar_texto)
df_latin_norm['objetivo'] = df_latin_norm['objetivo'].map(normalizar_texto)
df_latin_norm['par'] = 'la-en'

# 2. Normalizar Greek-English (usando df_nuevo que ya está extraído)
df_greek_norm = df_nuevo[['Greek', 'English']].copy()
df_greek_norm.rename(columns={'Greek': 'objetivo', 'English': 'fuente'}, inplace=True)
df_greek_norm['fuente'] = df_greek_norm['fuente'].map(normalizar_texto)
df_greek_norm['objetivo'] = df_greek_norm['objetivo'].map(normalizar_texto)
df_greek_norm['par'] = 'el-en'

display(df_latin_norm.head(2))
display(df_greek_norm.head(2))

C:\Users\jeffr\AppData\Local\Temp\ipykernel_19512\533999086.py:3: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_latin_norm.rename(columns={'la': 'fuente', 'en': 'objetivo'}, inplace=True)
C:\Users\jeffr\AppData\Local\Temp\ipykernel_19512\533999086.py:10: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_greek_norm.rename(columns={'Greek': 'fuente', 'English': 'objetivo'}, inplace=True)


,fuente,objetivo,par
0,post haec ambulabat iesus in galilaeam non enim volebat in iudaeam ambulare quia quaerebant eum iudaei interficere,"after these things, jesus walked in galilee: for he would not walk in judea, because the jews sought to kill him.",la-en
1,"despectus tibi sum nec, qui sim, quaeris, alexi, quam dives pecoris, nivei quam lactis abundans: mille meae siculis errant in montibus agnae; lac mihi non aestate novum, non frigore defit.","you scorn me, alexis, and ask not what i am'how rich in cattle, how wealthy in snow-white milk! a thousand lambs of mine roam over the sicilian hills; new milk fails me not, summer or winter.",la-en


,fuente,objetivo,par
0,επαvάληψη της συvσδoυ,resumption of the session,el-en
1,"κηρύσσω την επανάληψη της συνόδου του ευρωπαϊκού κοινοβουλίου η οποία είχε διακοπεί την παρασκευή 17 δεκεμβρίου και σας απευθύνω ξανά τις θερμές ευχές μου, ελπίζοντας να περάσατε καλά στις διακοπές.","i declare resumed the session of the european parliament adjourned on friday 17 december 1999, and i would like once again to wish you a happy new year in the hope that you enjoyed a pleasant festive period.",el-en


In [101]:
def create_hf_dataset_dict(data_df, seed=42):
    # Deduplicar ANTES de dividir para evitar fuga entre particiones
    data_df = data_df.dropna(subset=["fuente", "objetivo"])
    data_df = data_df.drop_duplicates(subset=["fuente", "objetivo"]).reset_index(drop=True)

    train_df, temp_df = train_test_split(
        data_df, test_size=0.20, random_state=seed, shuffle=True
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=seed, shuffle=True
    )

    return DatasetDict({
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "validation": Dataset.from_pandas(val_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False),
    })

# Aplicar el pipeline a ambos datasets normalizados
dataset_la_en = create_hf_dataset_dict(df_latin_norm)
dataset_el_en = create_hf_dataset_dict(df_greek_norm)

print("=== Latin-English Dataset ===")
print(dataset_la_en)

print("\n=== Greek-English Dataset ===")
print(dataset_el_en)

=== Latin-English Dataset ===
DatasetDict({
    train: Dataset({
        features: ['fuente', 'objetivo', 'par'],
        num_rows: 79180
    })
    validation: Dataset({
        features: ['fuente', 'objetivo', 'par'],
        num_rows: 9897
    })
    test: Dataset({
        features: ['fuente', 'objetivo', 'par'],
        num_rows: 9898
    })
})

=== Greek-English Dataset ===
DatasetDict({
    train: Dataset({
        features: ['fuente', 'objetivo', 'par'],
        num_rows: 1210004
    })
    validation: Dataset({
        features: ['fuente', 'objetivo', 'par'],
        num_rows: 151251
    })
    test: Dataset({
        features: ['fuente', 'objetivo', 'par'],
        num_rows: 151251
    })
})


**Verificar si existen elementos nulos o duplicados**

In [102]:
def verificar_nulos(dataset):
  for particion in dataset:
      df = dataset[particion].to_pandas()

      nulos = df.isnull().sum().sum()
      duplicados = df.duplicated().sum()

      print(f"\nPartición: {particion}")
      print(f"Registros nulos: {nulos}")
      print(f"Registros duplicados: {duplicados}")

# Valores nulos de Latin - English dataset
print("Valores nulos para el dataset Latin-English:")
verificar_nulos(dataset_la_en)

# Valores nulos de Greek - English dataset
print("\nValores nulos para el dataset Greek-English:")
verificar_nulos(dataset_el_en)

Valores nulos para el dataset Latin-English:

Partición: train
Registros nulos: 0
Registros duplicados: 0

Partición: validation
Registros nulos: 0
Registros duplicados: 0

Partición: test
Registros nulos: 0
Registros duplicados: 0

Valores nulos para el dataset Greek-English:

Partición: train
Registros nulos: 0
Registros duplicados: 0

Partición: validation
Registros nulos: 0
Registros duplicados: 0

Partición: test
Registros nulos: 0
Registros duplicados: 0


In [103]:
print("Dimensiones finales")
print("Latin-English Dataset: ", dataset_la_en.num_rows)
print("Greek-English Dataset: ", dataset_el_en.num_rows)

Dimensiones finales
Latin-English Dataset:  {'train': 79180, 'validation': 9897, 'test': 9898}
Greek-English Dataset:  {'train': 1210004, 'validation': 151251, 'test': 151251}


In [104]:
def revisar_fuga(dataset, columna="fuente"):
    train = set(dataset["train"][columna])
    for particion in ["validation", "test"]:
        textos = dataset[particion][columna]
        repetidos = sum(1 for t in textos if t in train)
        print(
            f"{particion}: {repetidos}/{len(textos)} "
            f"({repetidos / len(textos) * 100:.2f}%) de '{columna}' aparecen también en train"
        )


revisar_fuga(dataset_la_en)
revisar_fuga(dataset_el_en)

validation: 13/9897 (0.13%) de 'fuente' aparecen también en train
test: 10/9898 (0.10%) de 'fuente' aparecen también en train
validation: 5205/151251 (3.44%) de 'fuente' aparecen también en train
test: 5215/151251 (3.45%) de 'fuente' aparecen también en train


In [105]:
def palabras_mezcla_alfabetos(textos, max_ejemplos=10):
    patron_griego = re.compile(r"[\u0370-\u03FF\u1F00-\u1FFF]")
    patron_latino = re.compile(r"[A-Za-z]")
    total, mezcla, ejemplos = 0, 0, []

    for texto in textos:
        for palabra in str(texto).split():
            total += 1
            if patron_griego.search(palabra) and patron_latino.search(palabra):
                mezcla += 1
                if len(ejemplos) < max_ejemplos:
                    ejemplos.append(palabra)

    print(f"Palabras con mezcla de alfabetos: {mezcla}/{total} ({mezcla / total * 100:.3f}%)")
    print("Ejemplos:", ejemplos)


muestra_el = dataset_el_en["train"].shuffle(seed=42).select(range(50_000))
palabras_mezcla_alfabetos(muestra_el["fuente"])

Palabras con mezcla de alfabetos: 2007/1885967 (0.106%)
Ejemplos: ['eπίσης,', 'cομ(1999)', '(cοs))', 'aυτό', 'tσετσενία.', 'oι', 'kοινοβούλιο', 'eυρωπαϊκή', 'αlava', 'kατά']


**Procesar el texto, que quede listo para el modelo**

In [106]:
# Funcion para entrenar el tokenizador

def entrenar_tokenizador(df_train, nombre, vocab_size, max_oraciones=None):
    textos = pd.concat([
        df_train["fuente"],
        df_train["objetivo"]
    ]).dropna().astype(str)

    textos = textos.str.strip()
    textos = textos[textos != ""]

    # Para griego (1.2M de pares) conviene limitar, por ejemplo max_oraciones=500_000
    if max_oraciones is not None and len(textos) > max_oraciones:
        textos = textos.sample(max_oraciones, random_state=42)

    spm.SentencePieceTrainer.train(
        sentence_iterator=iter(textos),
        model_prefix=nombre,
        model_type="bpe",
        vocab_size=vocab_size,
        character_coverage=1.0,
        normalization_rule_name="identity",
        byte_fallback=True,
        pad_id=3,
        unk_id=0,
        bos_id=1,
        eos_id=2,
        pad_piece="<pad>"
    )

    print(f"Tokenizador generado: {nombre}")

In [107]:
# Entrenar tokenizadores
entrenar_tokenizador(
    dataset_la_en["train"].to_pandas(),
    "tokenizer_ingles_latin",
    vocab_size=8000
)

entrenar_tokenizador(
    dataset_el_en["train"].to_pandas(),
    "tokenizer_ingles_griego",
    vocab_size=16000
)

Tokenizador generado: tokenizer_ingles_latin
Tokenizador generado: tokenizer_ingles_griego


In [108]:
# Cargar tokenizadores entrenados
sp_latin = spm.SentencePieceProcessor(
    model_file="tokenizer_ingles_latin.model"
)

sp_griego = spm.SentencePieceProcessor(
    model_file="tokenizer_ingles_griego.model"
)

**Pruebas de tokenizador**

In [109]:
def evaluar_tokenizador(sp, textos):
    total_tokens = 0
    total_palabras = 0
    tokens_byte = 0

    for texto in textos:
        texto = str(texto)
        piezas = sp.encode(texto, out_type=str)
        total_tokens += len(piezas)
        total_palabras += len(texto.split())
        tokens_byte += sum(
            1 for p in piezas if p.startswith("<0x") and p.endswith(">")
        )

    fertilidad = total_tokens / total_palabras if total_palabras > 0 else 0
    pct_byte = tokens_byte / total_tokens * 100 if total_tokens > 0 else 0

    print("Total de tokens:", total_tokens)
    print(f"Fertilidad (tokens/palabra): {fertilidad:.3f}")
    print(f"Tokens de byte (fallback): {tokens_byte} ({pct_byte:.4f}%)")

In [110]:
def _metricas_tokenizador(sp, textos):
    total_tokens, total_palabras, tokens_byte = 0, 0, 0
    for texto in textos:
        texto = str(texto)
        piezas = sp.encode(texto, out_type=str)
        total_tokens += len(piezas)
        total_palabras += len(texto.split())
        tokens_byte += sum(1 for p in piezas if p.startswith("<0x") and p.endswith(">"))
    fertilidad = total_tokens / total_palabras if total_palabras else 0
    pct_byte = tokens_byte / total_tokens * 100 if total_tokens else 0
    return fertilidad, pct_byte


def comparar_vocab_sizes(df_train, df_val, prefijo, tamanos, max_oraciones=None, n_val=20000):
    muestra = df_val.sample(min(n_val, len(df_val)), random_state=42)
    resultados = []

    for v in tamanos:
        nombre = f"{prefijo}_{v}"
        entrenar_tokenizador(df_train, nombre, v, max_oraciones)
        sp = spm.SentencePieceProcessor(model_file=f"{nombre}.model")

        fert_f, byte_f = _metricas_tokenizador(sp, muestra["fuente"])
        fert_o, byte_o = _metricas_tokenizador(sp, muestra["objetivo"])

        resultados.append({
            "vocab_size": v,
            "fertilidad_fuente": round(fert_f, 3),
            "%bytes_fuente": round(byte_f, 4),
            "fertilidad_en": round(fert_o, 3),
            "%bytes_en": round(byte_o, 4),
        })

    return pd.DataFrame(resultados)


# Latín: rápido
tabla_la = comparar_vocab_sizes(
    dataset_la_en["train"].to_pandas(),
    dataset_la_en["validation"].to_pandas(),
    "tok_la", tamanos=[4000, 8000, 16000],
)
print(tabla_la)

# Griego: limita las oraciones para que no tarde demasiado
tabla_el = comparar_vocab_sizes(
    dataset_el_en["train"].to_pandas(),
    dataset_el_en["validation"].to_pandas(),
    "tok_el", tamanos=[8000, 16000, 32000], max_oraciones=300_000,
)
print(tabla_el)

Tokenizador generado: tok_la_4000
Tokenizador generado: tok_la_8000
Tokenizador generado: tok_la_16000
   vocab_size  fertilidad_fuente  %bytes_fuente  fertilidad_en  %bytes_en
0        4000              2.012            0.0          1.526        0.0
1        8000              1.756            0.0          1.365        0.0
2       16000              1.551            0.0          1.260        0.0
Tokenizador generado: tok_el_8000
Tokenizador generado: tok_el_16000
Tokenizador generado: tok_el_32000
   vocab_size  fertilidad_fuente  %bytes_fuente  fertilidad_en  %bytes_en
0        8000              1.565         0.0007          1.364     0.0008
1       16000              1.346         0.0008          1.233     0.0009
2       32000              1.220         0.0009          1.166     0.0009


In [ ]:
# Tokenizar los datasets
def tokenizar_dataset(dataset, sp, direccion="en->x"):
    # en->x: entrada = inglés ('objetivo'), salida = latín/griego ('fuente')
    # x->en: entrada = 'fuente', salida = 'objetivo' (comportamiento anterior)
    if direccion == "en->x":
        col_in, col_out = "objetivo", "fuente"
    elif direccion == "x->en":
        col_in, col_out = "fuente", "objetivo"
    else:
        raise ValueError("direccion debe ser 'en->x' o 'x->en'")

    def procesar_lote(lote):
        ids_in = sp.encode(lote[col_in])
        ids_out = sp.encode(lote[col_out])
        return {
            "input_ids": [[sp.bos_id()] + ids + [sp.eos_id()] for ids in ids_in],
            "labels": [[sp.bos_id()] + ids + [sp.eos_id()] for ids in ids_out],
        }

    return dataset.map(procesar_lote, batched=True)


def tokenizar_particiones(dataset, sp, direccion="en->x"):
    return {
        particion: tokenizar_dataset(dataset[particion], sp, direccion)
        for particion in ["train", "validation", "test"]
    }


def mostrar_ejemplo(ds, sp, indice=0, n=15, direccion="en->x"):
    col_in, col_out = ("objetivo", "fuente") if direccion == "en->x" else ("fuente", "objetivo")
    ej = ds[indice]
    print("Entrada:", ej[col_in])
    print("Salida esperada:", ej[col_out])
    print("input_ids:", ej["input_ids"][:n], "...")
    print("labels:", ej["labels"][:n], "...")
    print("input_ids -> piezas:", [sp.id_to_piece(i) for i in ej["input_ids"][:n]])
    print("labels -> piezas:", [sp.id_to_piece(i) for i in ej["labels"][:n]])
    print("¿Entrada reconstruida igual?", sp.decode(ej["input_ids"][1:-1]) == ej[col_in])
    print("¿Salida reconstruida igual?", sp.decode(ej["labels"][1:-1]) == ej[col_out])
tokenizado_la_en = tokenizar_particiones(dataset_la_en, sp_latin)
tokenizado_el_en = tokenizar_particiones(dataset_el_en, sp_griego)

Map:   0%|          | 0/79180 [00:00<?, ? examples/s]

Map:   0%|          | 0/9897 [00:00<?, ? examples/s]

Map:   0%|          | 0/9898 [00:00<?, ? examples/s]

Map:   0%|          | 0/1210004 [00:00<?, ? examples/s]

Map:   0%|          | 0/151251 [00:00<?, ? examples/s]

Map:   0%|          | 0/151251 [00:00<?, ? examples/s]

In [112]:
def mostrar_ejemplo(ds, sp, indice=0, n=15):
    ej = ds[indice]
    print("Fuente:", ej["fuente"])
    print("Objetivo:", ej["objetivo"])
    print("input_ids:", ej["input_ids"][:n], "...")
    print("labels:", ej["labels"][:n], "...")
    print("input_ids -> piezas:", [sp.id_to_piece(i) for i in ej["input_ids"][:n]])
    print("¿Fuente reconstruida igual?", sp.decode(ej["input_ids"][1:-1]) == ej["fuente"])
    print("¿Objetivo reconstruido igual?", sp.decode(ej["labels"][1:-1]) == ej["objetivo"])


mostrar_ejemplo(tokenizado_la_en["train"], sp_latin)
mostrar_ejemplo(tokenizado_el_en["train"], sp_griego)

Fuente: an non obsequereris principi civis, legatus imperatori, filius patri? ubi deinde disciplina, ubi mos a maioribus traditus, quodcumque imperator munus iniungeret, aequo animo paratoque subeundi? quid enim, si provincias ex provinciis ex bellis bella mandaret? apparet eodem illo uti iure, cum ad imperium revocet quo sit usus cum ad exercitum miserit, nihilque interesse ire legatum an redire principem iubeat, nisi quod maior sit obsequii gloria in eo, quod quis minus velit.
Objetivo: you were subject, legate and son: your duty lay in obedience to your ruler, your general and your father; else where would military discipline be, and the tradition of centuries of accepting readily, without question, whatever charges your general imposed? he could have assigned you province after province, sent you on campaign after campaign, for clearly he was exercising the same authority in recalling you to assume the imperial power as he did when he sent you out to command your army. an order is 

In [113]:
# Analizar longitudes y filtrar

def analizar_longitudes(tokenizado, nombre):
    print(f"\n{'=' * 40}\n{nombre} / TRAIN\n{'=' * 40}")
    for columna, etiqueta in [("input_ids", "Fuente"), ("labels", "Objetivo (inglés)")]:
        longitudes = np.array([len(x) for x in tokenizado["train"][columna]])
        print(f"\n{etiqueta}")
        print(f"  Mínimo: {longitudes.min()} | Máximo: {longitudes.max()}")
        print(f"  Media: {longitudes.mean():.2f} | Mediana: {np.median(longitudes):.1f}")
        for p in (90, 95, 99):
            print(f"  Percentil {p}: {np.percentile(longitudes, p):.0f}")


def filtrar_pares(ds, max_len=128, ratio_max=None):
    def conservar(ejemplo):
        ls, lt = len(ejemplo["input_ids"]), len(ejemplo["labels"])
        if ls > max_len or lt > max_len:
            return False
        if ratio_max is not None and max(ls, lt) / min(ls, lt) > ratio_max:
            return False
        return True

    antes = len(ds)
    ds = ds.filter(conservar)
    print(f"Conservados: {len(ds)}/{antes} ({len(ds) / antes * 100:.2f}%)")
    return ds


analizar_longitudes(tokenizado_la_en, "LATÍN - INGLÉS")
analizar_longitudes(tokenizado_el_en, "GRIEGO - INGLÉS")

# El filtro de razón de longitudes (pares mal alineados) solo en train
tokenizado_la_en["train"] = filtrar_pares(tokenizado_la_en["train"], max_len=128, ratio_max=3.0)
tokenizado_el_en["train"] = filtrar_pares(tokenizado_el_en["train"], max_len=128, ratio_max=3.0)

# En validation/test, solo el límite de longitud (para no facilitar artificialmente la evaluación)
for particion in ["validation", "test"]:
    tokenizado_la_en[particion] = filtrar_pares(tokenizado_la_en[particion], max_len=128)
    tokenizado_el_en[particion] = filtrar_pares(tokenizado_el_en[particion], max_len=128)


LATÍN - INGLÉS / TRAIN

Fuente
  Mínimo: 3 | Máximo: 338
  Media: 38.41 | Mediana: 32.0
  Percentil 90: 69
  Percentil 95: 87
  Percentil 99: 126

Objetivo (inglés)
  Mínimo: 3 | Máximo: 397
  Media: 47.47 | Mediana: 40.0
  Percentil 90: 86
  Percentil 95: 107
  Percentil 99: 157

GRIEGO - INGLÉS / TRAIN

Fuente
  Mínimo: 3 | Máximo: 1217
  Media: 52.84 | Mediana: 39.0
  Percentil 90: 108
  Percentil 95: 143
  Percentil 99: 228

Objetivo (inglés)
  Mínimo: 3 | Máximo: 1153
  Media: 48.25 | Mediana: 36.0
  Percentil 90: 98
  Percentil 95: 130
  Percentil 99: 206


Filter:   0%|          | 0/79180 [00:00<?, ? examples/s]

Conservados: 77189/79180 (97.49%)


Filter:   0%|          | 0/1210004 [00:00<?, ? examples/s]

Conservados: 1101425/1210004 (91.03%)


Filter:   0%|          | 0/9897 [00:00<?, ? examples/s]

... (3 outputs left out)

Filter:   0%|          | 0/9898 [00:00<?, ? examples/s]

Conservados: 9659/9898 (97.59%)


Filter:   0%|          | 0/151251 [00:00<?, ? examples/s]

Conservados: 138912/151251 (91.84%)


In [114]:
# Guardar tokenizado

def guardar_tokenizado(tokenizado, carpeta):
    for particion, ds in tokenizado.items():
        ds.save_to_disk(f"{carpeta}/{particion}")


def cargar_tokenizado(carpeta):
    return {p: load_from_disk(f"{carpeta}/{p}") for p in ["train", "validation", "test"]}


guardar_tokenizado(tokenizado_la_en, "datos/la_en")
guardar_tokenizado(tokenizado_el_en, "datos/el_en")

Saving the dataset (0/1 shards):   0%|          | 0/77189 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9660 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9659 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/1101425 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/139076 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/138912 [00:00<?, ? examples/s]

In [115]:
# 1. Cargar los tokenizadores elegidos
sp_latin = spm.SentencePieceProcessor(model_file="tok_la_8000.model")
sp_griego = spm.SentencePieceProcessor(model_file="tok_el_16000.model")

for nombre, sp in [("latín", sp_latin), ("griego", sp_griego)]:
    print(
        f"{nombre}: vocab={sp.get_piece_size()} | pad={sp.pad_id()} "
        f"unk={sp.unk_id()} bos={sp.bos_id()} eos={sp.eos_id()}"
    )

# 2. Retokenizar desde los datasets normalizados y deduplicados
tokenizado_la_en = tokenizar_particiones(dataset_la_en, sp_latin)
tokenizado_el_en = tokenizar_particiones(dataset_el_en, sp_griego)

# 3. Filtrar: razón de longitudes solo en train; max_len en todas las particiones
MAX_LEN = {"la": 160, "el": 128}

for clave, tokenizado in [("la", tokenizado_la_en), ("el", tokenizado_el_en)]:
    print(f"\n--- Filtrando {clave} (max_len={MAX_LEN[clave]}) ---")
    tokenizado["train"] = filtrar_pares(
        tokenizado["train"], max_len=MAX_LEN[clave], ratio_max=3.0
    )
    for particion in ["validation", "test"]:
        tokenizado[particion] = filtrar_pares(
            tokenizado[particion], max_len=MAX_LEN[clave]
        )

# 4. Comprobar un ejemplo real de cada uno
mostrar_ejemplo(tokenizado_la_en["train"], sp_latin)
mostrar_ejemplo(tokenizado_el_en["train"], sp_griego)

# 5. Guardar
guardar_tokenizado(tokenizado_la_en, "datos/la_en")
guardar_tokenizado(tokenizado_el_en, "datos/el_en")

latín: vocab=8000 | pad=3 unk=0 bos=1 eos=2
griego: vocab=16000 | pad=3 unk=0 bos=1 eos=2


Map:   0%|          | 0/79180 [00:00<?, ? examples/s]

Map:   0%|          | 0/9897 [00:00<?, ? examples/s]

Map:   0%|          | 0/9898 [00:00<?, ? examples/s]

Map:   0%|          | 0/1210004 [00:00<?, ? examples/s]

Map:   0%|          | 0/151251 [00:00<?, ? examples/s]

... (16 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/9812 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/1101526 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/139093 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/138900 [00:00<?, ? examples/s]